In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, auc
import os
import pickle

import scanpy as sc 
import json
import re
import pyranges as pr
from cellgrn.utils import enhancer_eval, eval_gene_peak2, eval_tf_recovery, eval_tf_gene, load_scenic2, load_linger_ctx,load_linger_all,load_thres_grn,eval_tf_recovery_ctx


import seaborn as sns


/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [ ]:
result_path = "/home/shaliu_fu/multireg/benchmark/output/melanoma/"
# dataset_config = json.load(open("/home/shaliu_fu/multireg/benchmark/bench_dataset/melanoma/RawData.json"))

soft_path = {

    "SCENIC+":"scenic2_output",
    "LINGER":"LINGER_output"

}

soft_load = {
    "SCENIC+":load_scenic2,
    "LINGER":load_linger_all,

}


In [3]:

# 0. load files 
# soft_res = {}
# for soft in soft_path.keys():
#     print(soft)
#     gene_peak_res, grn_res, tf_peak_res = soft_load[soft](result_path, soft_path[soft])
#     soft_res[soft] = {}
#     soft_res[soft]['grn_res'] = grn_res
#     soft_res[soft]['gene_peak_res'] = gene_peak_res
#     soft_res[soft]['tf_peak_res'] = tf_peak_res

# # ctx specific results
# soft_res_ctx = {}
# gene_peak_res, grn_res, tf_peak_res = load_linger_ctx(result_path, "LINGER_output")
# soft_res_ctx["LINGER_ctx"] = {}
# soft_res_ctx["LINGER_ctx"]['grn_res'] = grn_res
# soft_res_ctx["LINGER_ctx"]['gene_peak_res'] = gene_peak_res
# soft_res_ctx["LINGER_ctx"]['tf_peak_res'] = tf_peak_res
# with open("./melanoma_bench_methods.pkl", "wb") as f:
#     pickle.dump(soft_res, f)

# with open("./melanoma_bench_methods_ctx.pkl", "wb") as f:
#     pickle.dump(soft_res_ctx, f)

with open("./melanoma_bench_methods.pkl", "rb") as f:
    soft_res = pickle.load(f)
with open("./melanoma_bench_methods_ctx.pkl", "rb") as f:
    soft_res_ctx = pickle.load(f)

In [4]:
# soft_res['LINGER']['grn_res']

In [ ]:

for suffix in ["scale2"]:
    soft = f'linger_thres_{suffix}_samp'
    gene_peak_res, grn_res,tf_peak_res = load_thres_grn(f"/home/shaliu_fu/multireg/cellGRN/output/res_melanoma_linger/",
                                                        scale="sample",suffix=suffix)
    soft_res[soft] = {}
    soft_res[soft]['grn_res'] = grn_res
    soft_res[soft]['gene_peak_res'] = gene_peak_res
    soft_res[soft]['tf_peak_res'] = tf_peak_res

    soft = f'linger_thres_{suffix}_ctx'
    gene_peak_res, grn_res,tf_peak_res = load_thres_grn(f"/home/shaliu_fu/multireg/cellGRN/output/res_melanoma_linger/",
                                                        scale='celltype',suffix=suffix)
    soft_res_ctx[soft] = {}
    soft_res_ctx[soft]['grn_res'] = grn_res
    soft_res_ctx[soft]['gene_peak_res'] = gene_peak_res
    soft_res_ctx[soft]['tf_peak_res'] = tf_peak_res


    soft = f'scenic2_thres_{suffix}_samp'
    gene_peak_res, grn_res,tf_peak_res = load_thres_grn(f"/home/shaliu_fu/multireg/cellGRN/output/res_melanoma_scenic2/",
                                                        scale="sample",suffix=suffix)
    soft_res[soft] = {}
    soft_res[soft]['grn_res'] = grn_res
    soft_res[soft]['gene_peak_res'] = gene_peak_res
    soft_res[soft]['tf_peak_res'] = tf_peak_res



In [6]:
outdir = "/home/shaliu_fu/multireg/cellGRN/eval/results/melanoma/"
os.system(f"mkdir -p {outdir}")

# soft_res['linger_thres_scale2_samp']['gene_peak_res'].head()

0

In [ ]:
# 1. TF recovery: 
scRNA_tab = sc.read_h5ad("/home/shaliu_fu/multireg/cellGRN/data/melanoma/Melanoma-cell_line-RNA-counts.h5ad")
all_genes = set(scRNA_tab.var.index.values)

known_tf = {
    "MEL": ["MITF", "SOX10", "SNAI2", "ZEB2", "TFAP2A", "PAX3", "MXI1", "IRF4"],
    "INT": ["EGR3", "SOX6","NFATC2","ELF1", "ETV4"],
    "MES": ["TWIST1","ZEB1","SOX9","TCF7L2","SNAI1","JUN","FOSL1","GLI1","ETS1","PRRX2"]
}

cell_types = known_tf.keys()
tf_recovery_all = pd.DataFrame()
dataset_qc = pd.DataFrame()

for ctx in cell_types:
    
    sel_TF_ = set(known_tf[ctx])
    sel_TF = sel_TF_ & all_genes
    print(sel_TF_ - sel_TF)

    dataset_qc = pd.concat([dataset_qc,pd.DataFrame({"ctx":[ctx],
                                                     "all_TF":[len(sel_TF_)],
                                                     "dataset_TF":[len(sel_TF)]})],axis=0)
 

set()
set()
set()


In [ ]:
# common_genes = list(known_tf['MEL']+known_tf['MES']+known_tf['INT'])
common_genes = scRNA_tab.var_names
expression_matrix = scRNA_tab.X.toarray() if hasattr(scRNA_tab.X, "toarray") else scRNA_tab.X


gene_indices = [scRNA_tab.var_names.get_loc(gene) for gene in common_genes]
selected_genes_expression = expression_matrix[:, gene_indices]  # shape: (cells, len(common_genes))

expression_matrix_zscore = (expression_matrix - np.mean(expression_matrix, axis=0)) / np.std(expression_matrix, axis=0)
selected_genes_zscore = (selected_genes_expression - np.mean(selected_genes_expression, axis=0)) / np.std(selected_genes_expression, axis=0)

correlation_matrix = np.dot(expression_matrix_zscore.T, selected_genes_zscore) / expression_matrix.shape[0]

correlation_df = pd.DataFrame(correlation_matrix, index=scRNA_tab.var_names, columns=common_genes)

melted_df = correlation_df.reset_index().melt(id_vars="index", var_name="TF", value_name="Score")

melted_df.rename(columns={"index": "Gene"}, inplace=True)
melted_df = melted_df[["TF","Gene","Score"]]
melted_df.drop_duplicates(inplace=True)



In [16]:
print("hello")

hello


In [ ]:


rec_summary = pd.DataFrame()

rec_ctx = pd.DataFrame() 

melted_df = melted_df[melted_df['TF']!=melted_df['Gene']]
tf_baseline = melted_df.nlargest(10000,"Score")

tf_rec_ctx, tf_rec_base = eval_tf_recovery(grn_res=tf_baseline,gold_data=known_tf,label="Pearson",log=False) # ctx dataframe

rec_summary = pd.concat([rec_summary,tf_rec_base],axis=0)
tf_rec_ctx['method'] = "Pearson"
rec_ctx = pd.concat([rec_ctx, tf_rec_ctx],axis=0)

for soft in soft_res.keys():
# for soft in ["LINGER"]:
    tf_gene_res = soft_res[soft]['grn_res'].copy()
    tf_gene_res = tf_gene_res[["TF","Gene","Score"]]
    tf_gene_res = tf_gene_res.groupby(['TF', 'Gene'])['Score'].max().reset_index() #只保留最高的。
    # tf_gene_res.drop_duplicates(inplace=True) # 合并各个ctx结果
    if tf_gene_res is not None:
        
        tf_gene_res2 = tf_gene_res.nlargest(10000,"Score")

        tf_rec_ctx, tf_rec_res = eval_tf_recovery(grn_res=tf_gene_res2,gold_data=known_tf,label=soft,log=False) # ctx dataframe

        rec_summary = pd.concat([rec_summary,tf_rec_res],axis=0)
        tf_rec_ctx['method'] = soft
        rec_ctx = pd.concat([rec_ctx, tf_rec_ctx],axis=0)

for soft in soft_res_ctx.keys():
    tf_gene_res = soft_res_ctx[soft]['grn_res'].copy()

    tf_gene_res = tf_gene_res.groupby(['TF', 'Gene','cell_type'])['Score'].max().reset_index() 
    tf_gene_res.drop_duplicates(inplace=True) 
    if tf_gene_res is not None:       

        tf_rec_ctx, tf_rec_res = eval_tf_recovery_ctx(grn_res_ctx=tf_gene_res,gold_data=known_tf,label=soft,log=False) # ctx dataframe

        rec_summary = pd.concat([rec_summary,tf_rec_res],axis=0)
        
        tf_rec_ctx['method'] = soft
        rec_ctx = pd.concat([rec_ctx, tf_rec_ctx],axis=0)
        
rec_summary.to_csv(f"{outdir}/melanoma_tf_recov_tf_gene.csv",header=True,index=True)

rec_ctx.to_csv(f"{outdir}/melanoma_tf_recov_tf_gene_ctx.csv",header=True,index=True)

In [ ]:

rec_summary = pd.DataFrame()
rec_ctx = pd.DataFrame() 

for soft in soft_res.keys():

    tf_peak_res = soft_res[soft]['tf_peak_res']
    if tf_peak_res is not None:
        tf_peak_res = tf_peak_res[["TF","Peak","Score"]].copy()
        tf_peak_res = tf_peak_res.groupby(['TF', 'Peak'])['Score'].max().reset_index() #只保留最高的。
        
        tf_peak_res2 = tf_peak_res.nlargest(20000,"Score")
        tf_rec_ctx, tf_rec_res = eval_tf_recovery(grn_res=tf_peak_res2,gold_data=known_tf,label=soft,log=False) # ctx dataframe
        rec_summary = pd.concat([rec_summary,tf_rec_res],axis=0)
        tf_rec_ctx['method'] = soft
        rec_ctx = pd.concat([rec_ctx, tf_rec_ctx],axis=0)

for soft in soft_res_ctx.keys():

    tf_peak_res = soft_res_ctx[soft]['tf_peak_res'].copy()
    
    if tf_peak_res is not None: 

        tf_rec_ctx, tf_rec_res = eval_tf_recovery_ctx(grn_res_ctx=tf_peak_res,gold_data=known_tf,label=soft,log=False) # ctx dataframe
        rec_summary = pd.concat([rec_summary,tf_rec_res],axis=0)
        tf_rec_ctx['method'] = soft
        rec_ctx = pd.concat([rec_ctx, tf_rec_ctx],axis=0)


In [23]:
rec_summary.to_csv(f"{outdir}/melanoma_tf_recov_tf_peak.csv",header=True,index=True)
rec_ctx.to_csv(f"{outdir}/melanoma_tf_recov_tf_peak_ctx.csv",header=True,index=True)

In [9]:
# 2. TF-gene recovery: 

TF_gene_knock = pd.read_csv("/home/shaliu_fu/multireg/benchmark/datasets/melanoma/TF_gene_gold.txt",sep="\t",index_col=None,header=0)

TF_gene_knock2 = TF_gene_knock[TF_gene_knock['TF'].isin(all_genes)][TF_gene_knock['Gene'].isin(all_genes)]


melanoma_TF_knock_all = TF_gene_knock.apply(lambda x: f"{x[0]}_{x[1]}",axis=1)
melanoma_TF_knock_pair = TF_gene_knock2.apply(lambda x: f"{x[0]}_{x[1]}",axis=1)

dataset_qc = pd.DataFrame({"ctx":["melanoma"],"all_pairs":[len(set(melanoma_TF_knock_all))],"dataset_pair":[len(set(melanoma_TF_knock_pair))]})

/tmp/ipykernel_1512694/3861809147.py:8: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  melanoma_TF_knock_all = TF_gene_knock.apply(lambda x: f"{x[0]}_{x[1]}",axis=1)
/tmp/ipykernel_1512694/3861809147.py:9: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  melanoma_TF_knock_pair = TF_gene_knock2.apply(lambda x: f"{x[0]}_{x[1]}",axis=1)


In [ ]:


common_genes = [gene for gene in TF_gene_knock['TF'] if gene in scRNA_tab.var_names]
expression_matrix = scRNA_tab.X.toarray() if hasattr(scRNA_tab.X, "toarray") else scRNA_tab.X

gene_indices = [scRNA_tab.var_names.get_loc(gene) for gene in common_genes]
selected_genes_expression = expression_matrix[:, gene_indices]  # shape: (cells, len(common_genes))

expression_matrix_zscore = (expression_matrix - np.mean(expression_matrix, axis=0)) / np.std(expression_matrix, axis=0)
selected_genes_zscore = (selected_genes_expression - np.mean(selected_genes_expression, axis=0)) / np.std(selected_genes_expression, axis=0)

correlation_matrix = np.dot(expression_matrix_zscore.T, selected_genes_zscore) / expression_matrix.shape[0]

correlation_df = pd.DataFrame(correlation_matrix, index=scRNA_tab.var_names, columns=common_genes)

melted_df = correlation_df.reset_index().melt(id_vars="index", var_name="TF", value_name="Score")

melted_df.rename(columns={"index": "Gene"}, inplace=True)
melted_df = melted_df[["TF","Gene","Score"]]
melted_df.drop_duplicates(inplace=True)

In [11]:
TF_gene_knock2=TF_gene_knock2[TF_gene_knock2['TF']!=TF_gene_knock2['Gene']]
tf_knock_gold  =TF_gene_knock2.groupby(TF_gene_knock2.columns[0])[TF_gene_knock2.columns[1]].apply(list).to_dict()

In [13]:
tf_knock_gold2 = []
for tfs in tf_knock_gold.keys():
    tmp = [f"{tfs}_{i}" for i in tf_knock_gold[tfs]]
    tf_knock_gold2 = tf_knock_gold2+ tmp
sel_tfs = tf_knock_gold.keys()
tf_baseline = melted_df

In [22]:
pr_curve = pd.DataFrame()
res_summary = []
# 评估tf-gene

# for t_gold in tf_knock_gold.keys():
# for t_gold in ["NFATC2"]:
# t_gold = 'melanoma'
tf_knock = tf_knock_gold2
tf_baseline2 = tf_baseline[tf_baseline["TF"].isin(sel_tfs)]
pr_auc,epr,f1, pr_table = eval_tf_gene(grn_res=tf_baseline2.nlargest(10000,"Score"),gold_data=tf_knock,
                all_comb=tf_baseline2.shape[0],label="Pearson" ) # 
pr_curve = pd.concat([pr_curve,pr_table],axis=0)
res_summary.append(["Pearson", round(pr_auc,5), round(epr,5), round(f1,5)])
# pr_summary.append(["Pearson",t_gold, round(pr_auc,5)])
# epr_summary.append(["Pearson",t_gold, round(epr,5)])
for soft in soft_res.keys():
# for soft in ['FigR']:

# for soft in spa_res.keys():
#     gene_peak_res = spa_res[soft]
    tf_gene_res = soft_res[soft]['grn_res']
    if tf_gene_res is not None:
        tf_gene_res = tf_gene_res[["TF","Gene","Score"]]
        tf_gene_res = tf_gene_res.groupby(['TF', 'Gene'])['Score'].max().reset_index() #只保留最高的。
        
        # tf_gene_res2 = tf_gene_res.nlargest(10000,"Score")
        tf_gene_res2 = tf_gene_res[tf_gene_res['TF'].isin(sel_tfs)]
        tf_gene_res2 =  tf_gene_res2.nlargest(10000,"Score")
        # print(f"{t_gold}: {soft}- candidates : {tf_gene_res2.shape[0]}")
        if tf_gene_res2.shape[0] > 1:
            pr_auc,epr,f1, pr_table = eval_tf_gene(grn_res=tf_gene_res2,gold_data=tf_knock,
                        all_comb=tf_baseline2.shape[0],label=soft ) # 
        else:
            pr_auc = 0
            epr = 0
            f1 = 0
            pr_table = None
        pr_curve = pd.concat([pr_curve,pr_table],axis=0)

        res_summary.append([soft, round(pr_auc,5), round(epr,5), round(f1,5)])

res_summary = pd.DataFrame(res_summary)
res_summary.columns = ["method","PRAUC","EPR","F1"]

/tmp/ipykernel_1512694/181237394.py:6: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  soft_pred['pair'] = soft_pred.apply(lambda row: f"{row[0]}_{row[1]}", axis=1)
/tmp/ipykernel_1512694/181237394.py:6: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  soft_pred['pair'] = soft_pred.apply(lambda row: f"{row[0]}_{row[1]}", axis=1)
/tmp/ipykernel_1512694/181237394.py:6: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  soft_pred['pair'] = soft_

In [ ]:



out_res = res_summary
# out_res = out_res[out_res['dataset']=="melanoma"]

out_res.to_csv(f"{outdir}/melanoma_tf_gene_knock_res.csv",index=False,header=True)

pr_curve.to_csv(f"{outdir}/melanoma_tf_gene_knock_pr_curve.csv",index=False,header=True)


In [ ]:

mpra_mel_gold =  pd.read_csv("/home/shaliu_fu/multireg/benchmark/datasets/melanoma/mpra_mel_cre.bed",sep='\t',header=None)
mpra_mel_gold['Peak'] = mpra_mel_gold.apply(lambda row:f"{row[0]}:{row[1]}-{row[2]}", axis=1)

mpra_mes_gold =  pd.read_csv("/home/shaliu_fu/multireg/benchmark/datasets/melanoma/mpra_mes_cre.bed",sep='\t',header=None)
mpra_mes_gold['Peak'] = mpra_mes_gold.apply(lambda row:f"{row[0]}:{row[1]}-{row[2]}", axis=1)

mpra_gold = mpra_mel_gold.copy()
mpra_gold[3] = mpra_mel_gold[3]+mpra_mes_gold[3]

In [ ]:

# mpra_gold = pd.concat([mpra_mel_gold, mpra_mes_gold],axis=0)


pr_curve = pd.DataFrame()
res_summary = []
for soft in soft_res.keys():
    gene_peak_res = soft_res[soft]['gene_peak_res'].copy()

    if gene_peak_res is not None:
        pr_table,pr_auc,epr,f1  = enhancer_eval(mpra_gold, gene_peak_res,soft)
        pr_curve = pd.concat([pr_curve,pr_table],axis=0)
        res_summary.append([soft,round(pr_auc,5),round(epr,5),round(f1,5)])

res_summary = pd.DataFrame(res_summary)
res_summary.columns = ["method","PR-AUC","EPR","F1"]

In [ ]:

out_res = res_summary
out_res.to_csv(f"{outdir}/melanoma_gene_peak_mpra_enhancer_res.csv",index=False,header=True)

pr_curve.to_csv(f"{outdir}/melanoma_gene_peak_mpra_enhancer_pr_curve.csv",index=False,header=True)

In [ ]:
pr_curve = pd.DataFrame()
res_summary = []
for soft in soft_res.keys():
    tf_peak_res = soft_res[soft]['tf_peak_res']

    if tf_peak_res is not None:
        tf_peak_res = tf_peak_res.nlargest(20000,"Score")
        pr_table,pr_auc,epr,f1  = enhancer_eval(mpra_gold, tf_peak_res,soft)
        pr_curve = pd.concat([pr_curve,pr_table],axis=0)
        res_summary.append([soft,round(pr_auc,5),round(epr,5),round(f1,5)])

res_summary = pd.DataFrame(res_summary)
res_summary.columns = ["method","PR-AUC","EPR","F1"]

/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:225: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  label[label>1] = 1
/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:225: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  label[label>1] = 1
/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/cellgrn/utils.py:225: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-

In [ ]:

out_res = res_summary
# out_res['EPR'] = epr_summary['EPR']
out_res.to_csv(f"{outdir}/melanoma_tf_peak_mpra_enhancer_res.csv",index=False,header=True)

pr_curve.to_csv(f"{outdir}/melanoma_tf_peak_mpra_enhancer_pr_curve.csv",index=False,header=True)